[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gsilvaoelker/campos_ondas_electromagneticas/blob/main/unidad_02/07_fasores_y_ondas_planas.ipynb)

Pulse el botón para ejecutar este notebook en Google Colab sin instalar nada.

# Semana 7 — Fasores y ondas planas sin pérdidas

**Campos y Ondas Electromagnéticas (ICEE1033) — Unidad 2**

## 1. Objetivos de aprendizaje

Al terminar este notebook usted podrá:

1. Escribir una onda plana en forma de fasor y volver al tiempo cuando lo
   necesite.
2. Calcular la longitud de onda y la constante de fase a partir de la
   frecuencia.
3. Obtener $\mathbf{H}$ a partir de $\mathbf{E}$ usando la impedancia
   intrínseca del medio.
4. Relacionar el potencial vector $\mathbf{A}$ con los campos que produce.

In [ ]:
# Preparación del entorno: local o Google Colab, con verificación SHA256.
import hashlib
import sys
import urllib.request
from pathlib import Path

MODULOS = {
    "utilidades_notebook.py": "e1811892d086ca99e03694c0e70853886f63be58326e3e7232c6978db1fdf7a9",
    "constantes_fisicas.py": "394c39ad2aac2f4870620e3df6e276046f04abb811d1b28b6fa14f1836fe20ce",
    "ondas_planas.py": "64889dfed95d61988f17448266471dcdc8820b0ce51d919144ae37b7c10d8bb3",
}
URL_SRC = (
    "https://raw.githubusercontent.com/"
    "gsilvaoelker/campos_ondas_electromagneticas/main/src/"
)


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


candidatos = [Path.cwd(), *Path.cwd().parents]
raiz_repo = next((p for p in candidatos if (p / ".git").exists()), None)
if raiz_repo is not None:
    for modulo, esperado in MODULOS.items():
        archivo = raiz_repo / "src" / modulo
        if not archivo.exists() or sha256(archivo) != esperado:
            raise RuntimeError(
                f"Hash local desactualizado para {modulo}. "
                "Ejecute scripts/refresh_notebook_hashes.py."
            )
    raiz = raiz_repo
else:
    raiz = Path.cwd()
    (raiz / "src").mkdir(exist_ok=True)
    for modulo, esperado in MODULOS.items():
        destino = raiz / "src" / modulo
        if destino.exists() and sha256(destino) == esperado:
            continue
        with urllib.request.urlopen(URL_SRC + modulo, timeout=30) as respuesta:
            datos = respuesta.read()
        obtenido = hashlib.sha256(datos).hexdigest()
        if obtenido != esperado:
            raise RuntimeError(
                f"SHA256 inválido para {modulo}: {obtenido} != {esperado}"
            )
        destino.write_bytes(datos)

sys.path.insert(0, str(raiz / "src"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from constantes_fisicas import MU_0, IMPEDANCIA_VACIO
from ondas_planas import (
    numero_de_onda,
    longitud_de_onda,
    impedancia_intrinseca,
    campo_instantaneo,
)
from utilidades_notebook import configurar_estilo_graficos, tabla_resultados

configurar_estilo_graficos()

## 2. De dónde sale todo esto

### 2.1 Por qué usamos fasores

Casi todas las señales del curso son sinusoidales. Escribirlas con cosenos
y arrastrar identidades trigonométricas es tedioso y propenso a errores.

El truco del fasor es guardar solo la amplitud y la fase, y dejar el
$\cos(\omega t)$ implícito. Con eso, derivar respecto del tiempo se
convierte en multiplicar por $j\omega$, y las ecuaciones diferenciales de
Maxwell se vuelven ecuaciones algebraicas.

Es el mismo cambio de mirada que usted ya hizo en circuitos cuando pasó de
resolver ecuaciones diferenciales a multiplicar por impedancias.

### 2.2 Qué es una onda plana

Una onda plana es la solución más simple de la ecuación de onda: los
frentes de onda son planos infinitos perpendiculares a la dirección de
propagación, y sobre cada plano el campo vale lo mismo en todos los puntos.

Ninguna fuente real produce una onda plana perfecta. Pero lejos de
cualquier antena, en una región pequeña comparada con la distancia, la onda
se parece mucho a una plana. Por eso vale la pena estudiarla.

### 2.3 La impedancia intrínseca

En una onda plana, $\mathbf{E}$ y $\mathbf{H}$ no son independientes. Están
en fase, son perpendiculares entre sí, y el cociente entre sus magnitudes es
siempre el mismo número: la impedancia intrínseca del medio.

En el vacío ese número vale unos 377 $\Omega$. Es una propiedad del espacio,
no de la onda.

## 3. Ecuaciones

**Onda plana viajando en $+z$, en el tiempo y como fasor:**

$$
E_x(z,t) = E_0\cos(\omega t - \beta z + \phi_0)
\qquad\Longleftrightarrow\qquad
\tilde{E}_x(z) = E_0\,e^{j\phi_0}e^{-j\beta z}.
$$

**Constante de fase y longitud de onda:**

$$
\beta = \frac{\omega}{u} = \frac{2\pi}{\lambda},
\qquad
\lambda = \frac{u}{f},
\qquad
u = \frac{c}{\sqrt{\varepsilon_r\mu_r}}.
$$

**Impedancia intrínseca y campo magnético:**

$$
\eta = \sqrt{\frac{\mu}{\varepsilon}} = \eta_0\sqrt{\frac{\mu_r}{\varepsilon_r}},
\qquad
H_0 = \frac{E_0}{\eta},
\qquad
\eta_0 = \sqrt{\frac{\mu_0}{\varepsilon_0}} \approx 377~\Omega .
$$

**Potencial vector magnético.** Si $\mathbf{A}$ es una onda plana de
amplitud $A_0$ y frecuencia $\omega$, los campos que produce tienen
amplitudes

$$
|\mathbf{E}| = \omega A_0,
\qquad
|\mathbf{H}| = \frac{k A_0}{\mu_0},
\qquad k = \frac{\omega}{c}.
$$

El cociente entre ambas debe dar $\eta_0$; lo comprobaremos numéricamente.

## 4. Qué significa físicamente

**$\beta$ mide cuánta fase se acumula por metro.** Si $\beta = 21$ rad/m, la
onda gira 21 radianes de fase en cada metro que avanza. Recorrer una
longitud de onda completa equivale a girar exactamente $2\pi$.

**La frecuencia la fija la fuente; la longitud de onda, el medio.** Al
entrar a un dieléctrico la onda no cambia de frecuencia —eso sería violar la
continuidad en la interfaz— pero sí se acorta, porque viaja más lento.

**$E$ y $H$ están en fase y son perpendiculares.** Los dos alcanzan su
máximo en el mismo instante y en el mismo punto. Esto solo pasa en medios
sin pérdidas; en la semana 8 veremos que las pérdidas los desfasan.

**El potencial vector es un atajo de cálculo.** No se mide directamente,
pero es mucho más fácil calcular $\mathbf{A}$ a partir de las corrientes y
después derivar, que calcular $\mathbf{E}$ y $\mathbf{H}$ de una vez. Lo
volveremos a usar en la semana 14, con antenas.

## 5. Parámetros modificables

Esta es la única celda que conviene editar. Cambie un valor, ejecute el notebook
completo y compare con lo que tenía antes.

In [ ]:
# --- Problema 1: onda plana en el vacío ---
frecuencia = 1.0e9        # frecuencia de la onda [Hz]
amplitud_E = 10.0         # amplitud del campo eléctrico E_0 [V/m]
fase_inicial_grados = 30.0  # fase inicial phi_0 [grados]
z_evaluacion = 0.05       # posición donde evaluar el campo [m]
t_evaluacion = 0.2e-9     # instante donde evaluar el campo [s]

# --- Problema 2: potencial vector ---
frecuencia_A = 100.0e6    # frecuencia [Hz]
amplitud_A = 2.0e-8       # amplitud del potencial vector A_0 [Wb/m]

## 6. Implementación

### 6.1 Problema 1 — la onda plana

In [ ]:
beta = numero_de_onda(frecuencia)
longitud = longitud_de_onda(frecuencia)
eta = impedancia_intrinseca()
amplitud_H = amplitud_E / eta

fase_inicial = np.deg2rad(fase_inicial_grados)
campo_en_el_punto = campo_instantaneo(
    amplitud_E, frecuencia, z_evaluacion, t_evaluacion, fase_inicial
)

### 6.2 Problema 2 — campos a partir del potencial vector

In [ ]:
omega_A = 2.0 * np.pi * frecuencia_A
k_A = numero_de_onda(frecuencia_A)

campo_E_desde_A = omega_A * amplitud_A
campo_H_desde_A = k_A * amplitud_A / MU_0
cociente = campo_E_desde_A / campo_H_desde_A

## 7. Resultados numéricos

In [ ]:
tabla_resultados(
    [
        ("Longitud de onda", "lambda", longitud, "m"),
        ("Constante de fase", "beta", beta, "rad/m"),
        ("Impedancia intrínseca", "eta", eta, "ohm"),
        ("Amplitud del campo magnético", "H_0", amplitud_H, "A/m"),
        ("Campo en (z, t)", "E_x", campo_en_el_punto, "V/m"),
    ]
)

In [ ]:
tabla_resultados(
    [
        ("Número de onda", "k", k_A, "rad/m"),
        ("Campo eléctrico", "|E|", campo_E_desde_A, "V/m"),
        ("Campo magnético", "|H|", campo_H_desde_A, "A/m"),
        ("Cociente entre ambos", "|E|/|H|", cociente, "ohm"),
        ("Impedancia del vacío", "eta_0", IMPEDANCIA_VACIO, "ohm"),
    ]
)

In [ ]:
print(f"|E|/|H|  = {cociente:.6f} ohm")
print(f"eta_0    = {IMPEDANCIA_VACIO:.6f} ohm")
print(f"Error relativo = {abs(cociente / IMPEDANCIA_VACIO - 1.0):.3e}")

## 8. Visualización

Fotografía de la onda en un instante fijo. $E_x$ y $H_y$ se dibujan con
escalas distintas porque sus magnitudes difieren en un factor $\eta_0$.

In [ ]:
z = np.linspace(0.0, 2.0 * longitud, 500)
E_perfil = campo_instantaneo(amplitud_E, frecuencia, z, 0.0, fase_inicial)
H_perfil = E_perfil / eta

fig, eje_E = plt.subplots()
eje_E.plot(z * 100.0, E_perfil, color="tab:blue", label="E_x")
eje_E.set_xlabel("z (cm)")
eje_E.set_ylabel("E_x (V/m)", color="tab:blue")
eje_E.tick_params(axis="y", labelcolor="tab:blue")

eje_H = eje_E.twinx()
eje_H.plot(z * 100.0, H_perfil, color="tab:red", linestyle="--", label="H_y")
eje_H.set_ylabel("H_y (A/m)", color="tab:red")
eje_H.tick_params(axis="y", labelcolor="tab:red")
eje_H.grid(False)

eje_E.set_title("Onda plana sin pérdidas: E y H suben y bajan juntos")
fig.tight_layout()
plt.show()

## 9. Qué nos dicen los resultados

**Las dos curvas se superponen exactamente.** Cruzan el cero en los mismos
puntos y llegan al máximo a la vez. Eso es lo que significa "estar en fase",
y es la marca de un medio sin pérdidas.

**A 1 GHz la longitud de onda es de unos 30 cm.** Del tamaño de una regla.
Por eso a estas frecuencias el tamaño de un cable ya importa: deja de ser un
punto del circuito y pasa a ser una línea de transmisión, que es justamente
el tema de la Unidad 3.

**El campo magnético es numéricamente pequeño.** Con 10 V/m se obtienen unos
0.027 A/m. No es que el campo magnético sea débil: es que se mide en otras
unidades. El factor de conversión entre ambos es $\eta_0$.

**El potencial vector reproduce $\eta_0$.** Calculamos $|E|$ y $|H|$ por
caminos distintos, cada uno con sus propias constantes, y su cociente da
377 $\Omega$ con error del orden de $10^{-16}$. Es una buena señal de que la
formulación con potenciales es consistente con la de campos.

## 10. Ejercicios para experimentar

            1. Suba `frecuencia` a `2.4e9` (la banda de WiFi). ¿Cuánto mide ahora la
               longitud de onda? ¿Cabe dentro de una pieza?
            2. Baje `frecuencia` a `100e3` (radio de onda larga). ¿De qué tamaño tendría
               que ser una antena que mida media longitud de onda?
            3. Duplique `amplitud_E`. ¿Cambia $\beta$? ¿Cambia $H_0$? ¿Cambia $\eta$?
               Explique por qué unas cosas cambian y otras no.
            4. Ponga `fase_inicial_grados = 0.0` y observe el gráfico. ¿Dónde queda ahora
               el primer máximo?
            5. Cambie `t_evaluacion` a un cuarto de período (`1.0 / (4.0 * frecuencia)`).
               ¿Cuánto se movió
               el patrón?
            6. Multiplique `amplitud_A` por 10. ¿Cambia el cociente $|E|/|H|$? ¿Por qué
               ese cociente no depende de la amplitud?